# Task 4: 5-Fold SGKF Nightmare-Hardened 2:1 Vision Trio (Colab T4 Resilient & Fast-TTA)

### Progress & Completed Models
- ✅ **`cnxv3`** (`convnext_base.dinov3_lvd1689m` @ `192x384`, 5 folds COMPLETE): **`OOF_W6 F1 = 0.98878` | `LL = 0.05356` | `ExtHard F1 = 0.95465`** -> saved in `/content/drive/MyDrive/t4_hard_outputs/t4_hard_cnxv3.zip`.
- 🎯 **Remaining Trio Backbones (`MODELS_TO_RUN`):** **`dinov3l`** (faster ViT-L, `patch16` 288 tokens) first, then **`eva02l`** (`patch14`, `epochs=4`, hit `0.9712` by Ep 1).

### Key Upgrades in this Version
- **Epoch-Level Auto-Resume (`fold{k}_ckpt.pt`)**: saves trainable+EMA+optimizer+scheduler atomically after **every epoch** (auto-cleans after `fold{k}.npz` finishes) so Colab disconnects mid-fold lose zero completed epochs!
- **~5x Faster Post-Fold 6-View TTA Eval (`GPU-Vectorized`)**: transfers single-frame `uint8` images and runs rotation, `flip`, `interior_gw_stretch_torch`, and Nightmare photometric jitter directly on GPU with grad-checkpointing disabled during eval (eliminates the ~25 min/fold CPU bottleneck on 2 vCPUs).
- **Turing T4 `sm_75` Safe Runtime**: enforces `float16 + GradScaler + NCHW` + `probe_runtime_engine` preflight check + fixed `GradScaler` LR-scheduler step order (zero skipped-step warning).

### Quick Start to Resume Remaining Models
1. Select kernel -> `Colab` -> **`GPU` (`T4` / `L4` / `A100`)**.
2. If `/content/t4_data` was cleared after disconnect, make sure `task4_colab_pack_parts` is uploaded -> **Cell 2** re-extracts automatically.
3. Run **Run All** (`MODELS_TO_RUN = ["dinov3l", "eva02l"]` in **Cell 1** skips completed `cnxv3` and resumes automatically from the latest finished epoch/fold on Drive!).

In [4]:
# Cell 1: Dependencies (timm >= 1.0.20), GPU Precision Selection, Config & Drive Mount
import importlib, importlib.metadata, os, subprocess, sys

def parse_ver(v: str):
    parts = []
    for tok in v.split(".")[:3]:
        digits = "".join(ch for ch in tok if ch.isdigit())
        parts.append(int(digits) if digits else 0)
    return tuple(parts)

need_install = []
try:
    cur_timm = importlib.metadata.version("timm")
    if parse_ver(cur_timm) < (1, 0, 20):
        print(f"Upgrading old Colab timm {cur_timm} -> >=1.0.20 (required for DINOv3)...")
        need_install.append("timm>=1.0.20")
except importlib.metadata.PackageNotFoundError:
    need_install.append("timm>=1.0.20")

for pkg in ("scikit-learn", "pandas", "numpy", "pillow"):
    try:
        importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        need_install.append(pkg)

if need_install:
    had_stale_timm = "timm" in sys.modules
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", *need_install])
    importlib.invalidate_caches()
    if had_stale_timm:
        raise RuntimeError("Upgraded timm in-place; click Restart Kernel once and Run All!")

import numpy as np
import pandas as pd
import timm
import torch

print(f"Verified versions -> timm=={timm.__version__} | torch=={torch.__version__} | python=={sys.version.split()[0]}")
assert parse_ver(timm.__version__) >= (1, 0, 20), f"Restart kernel once! Loaded stale timm {timm.__version__}"

# --- Run Configuration ---
# cnxv3 is ALREADY COMPLETE in Drive (OOF_W6=0.98878, ExtHard=0.95465). Add "cnxv3" back anytime; completed folds auto-resume in <2s.
MODELS_TO_RUN = ["dinov3l", "eva02l"]
FREEZE_T4_LOWER = True     # On ViTs freezes blocks.0..(FREEZE_VIT_BLOCKS-1); never freezes patch_embed/tokens or ConvNeXt stem
FREEZE_VIT_BLOCKS = 6      # Keeps patch_embed + 18/24 upper blocks trainable (75% depth adapts to corruptions)
EPOCHS_OVERRIDE = None     # None uses calibrated defaults (dinov3l=5, eva02l=4, cnxv3=6 @ eff_bs=48)
MAX_FOLDS = 5
NUM_WORKERS = 2            # Free Colab T4 has 2 vCPUs
P_NIGHTMARE = 0.50         # Balanced 50% clean/flip + 50% nightmare corruptions
P_NORM_VIEW = 0.25         # Probability of interior Gray-World contrast stretch view during training
USE_DRIVE = True
SEED = 42

def select_amp_dtype(device: torch.device):
    """Use bfloat16 strictly on Ampere+ (cc >= 8.0); Turing T4 (sm_75) uses native float16 + GradScaler."""
    if device.type != "cuda":
        return None, False
    cc_major, _ = torch.cuda.get_device_capability(0)
    if cc_major >= 8 and torch.cuda.is_bf16_supported():
        return torch.bfloat16, False
    return torch.float16, True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu"))
AMP_DTYPE, USE_SCALER = select_amp_dtype(DEVICE)

if DEVICE.type == "cuda":
    p = torch.cuda.get_device_properties(0)
    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    torch.backends.cudnn.benchmark = (cc_major >= 8)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")
    print(
        f"GPU: {p.name} ({p.total_memory / (1024**3):.1f} GB VRAM) | cc={cc_major}.{cc_minor} | "
        f"torch_bf16_flag={torch.cuda.is_bf16_supported()} | selected_amp={AMP_DTYPE} | cpus={os.cpu_count()}"
    )
else:
    print(f"WARNING: Running on {DEVICE} (no CUDA GPU detected!)")

OUT_DIR = "/content/t4_hard_outputs"
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = "/content/drive/MyDrive/t4_hard_outputs"
    except Exception as exc:
        print(f"Drive mount skipped/unavailable ({exc}); saving locally to {OUT_DIR}")

os.makedirs(OUT_DIR, exist_ok=True)
import shutil as _shutil
_free_gb = _shutil.disk_usage(OUT_DIR).free / (1024**3)
print(f"Checkpoints & final zips will be saved to: {OUT_DIR} (free space: {_free_gb:.2f} GB)")
for _mk in MODELS_TO_RUN:
    _fd = os.path.join(OUT_DIR, f"folds_{_mk}")
    _done = sorted(f for f in os.listdir(_fd) if f.startswith("fold") and f.endswith(".npz")) if os.path.isdir(_fd) else []
    _ckpt = sorted(f for f in os.listdir(_fd) if f.endswith("_ckpt.pt")) if os.path.isdir(_fd) else []
    print(f"  Existing [{_mk}]: completed folds={_done} | epoch_ckpts={_ckpt}")

Verified versions -> timm==1.0.31 | torch==2.14.1+cu130 | python==3.13.16
GPU: Tesla T4 (14.6 GB VRAM) | cc=7.5 | torch_bf16_flag=True | selected_amp=torch.float16 | cpus=8
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Checkpoints & final zips will be saved to: /content/drive/MyDrive/t4_hard_outputs (free space: 223.07 GB)
  Existing [dinov3l]: completed folds=['fold0.npz', 'fold1.npz', 'fold2.npz', 'fold3.npz'] | epoch_ckpts=[]
  Existing [eva02l]: completed folds=[] | epoch_ckpts=[]


In [5]:
# Cell 2: Dataset Pack Verification & Extraction (/content/t4_data)
import glob, hashlib, subprocess, zipfile

DATA_DIR = "/content/t4_data"
os.makedirs(DATA_DIR, exist_ok=True)

EXPECTED_TAR_BYTES = 768022016
EXPECTED_TAR_SHA256 = "d7316deeb95c78ff230229c84f5a2ffa0cd86874a2753405ac9917ccc68a3a44"
EXPECTED_NUM_PARTS = 37
EXPECTED_STD_PART_BYTES = 20971520
EXPECTED_LAST_PART_BYTES = 13047296

def count_real_files(folder: str) -> int:
    """Count real JPGs while ignoring macOS ._ AppleDouble stub files."""
    if not os.path.isdir(folder):
        return 0
    return sum(1 for f in os.listdir(folder) if not f.startswith("._"))

def is_data_ready(d: str) -> bool:
    return (
        os.path.isfile(os.path.join(d, "folds_sgkf.npy"))
        and os.path.isfile(os.path.join(d, "train.csv"))
        and os.path.isfile(os.path.join(d, "test.csv"))
        and count_real_files(os.path.join(d, "train")) >= 31546
        and count_real_files(os.path.join(d, "test")) >= 25889
    )

def sha256_file(path: str):
    h = hashlib.sha256()
    total = 0
    with open(path, "rb") as f:
        while True:
            buf = f.read(8 * 1024 * 1024)
            if not buf:
                break
            h.update(buf)
            total += len(buf)
    return total, h.hexdigest()

def prepare_dataset_pack():
    if is_data_ready(DATA_DIR):
        print("Dataset already extracted and verified in", DATA_DIR)
        return

    merged_tar = "/content/task4_colab_pack.tar"
    joined_from_parts = False
    if os.path.isfile(merged_tar):
        cur_sz = os.path.getsize(merged_tar)
        if cur_sz != EXPECTED_TAR_BYTES or sha256_file(merged_tar)[1] != EXPECTED_TAR_SHA256:
            print("Removing incomplete/corrupt stale archive:", merged_tar)
            os.remove(merged_tar)

    parts_dirs = [
        "/content/task4_colab_pack_parts",
        "/task4_colab_pack_parts",
        "/content/drive/MyDrive/task4_colab_pack_parts",
    ]
    pdir = next((d for d in parts_dirs if os.path.isdir(d) and glob.glob(os.path.join(d, "part_*"))), None)
    parts_error = None
    if pdir and not os.path.isfile(merged_tar):
        expected_paths, bad_parts = [], []
        for idx in range(EXPECTED_NUM_PARTS):
            pname = f"part_{idx:02d}"
            exp_sz = EXPECTED_LAST_PART_BYTES if idx == EXPECTED_NUM_PARTS - 1 else EXPECTED_STD_PART_BYTES
            ppath = os.path.join(pdir, pname)
            if not os.path.isfile(ppath):
                bad_parts.append(f"{pname} (missing)")
            elif os.path.getsize(ppath) != exp_sz:
                bad_parts.append(f"{pname} ({os.path.getsize(ppath):,}B != {exp_sz:,}B)")
            else:
                expected_paths.append(ppath)
        if bad_parts:
            parts_error = (
                f"Upload incomplete in {pdir}: {EXPECTED_NUM_PARTS - len(bad_parts)}/{EXPECTED_NUM_PARTS} valid chunks. "
                f"Missing/mismatch ({len(bad_parts)}): {bad_parts[:8]}. "
                "Right-click project/task4/data/task4_colab_pack_parts -> 'Upload to Colab' and wait for completion."
            )
        else:
            print(f"Joining {len(expected_paths)} verified chunks from {pdir} -> {merged_tar} ...")
            h = hashlib.sha256()
            total = 0
            with open(merged_tar, "wb") as out_f:
                for pf in expected_paths:
                    with open(pf, "rb") as in_f:
                        while True:
                            buf = in_f.read(8 * 1024 * 1024)
                            if not buf:
                                break
                            out_f.write(buf)
                            h.update(buf)
                            total += len(buf)
            if total != EXPECTED_TAR_BYTES or h.hexdigest() != EXPECTED_TAR_SHA256:
                os.remove(merged_tar)
                raise RuntimeError(f"Joined archive checksum mismatch: bytes={total}, sha256={h.hexdigest()}")
            joined_from_parts = True
            print(f"Verified pack SHA-256: {EXPECTED_TAR_SHA256[:16]}... ({total / (1024**2):.1f} MiB)")

    direct_candidates = [
        merged_tar, "/content/task4_colab_pack.zip", "/task4_colab_pack.tar", "/task4_colab_pack.zip",
        "/content/drive/MyDrive/task4_colab_pack.tar", "/content/drive/MyDrive/task4_colab_pack.zip",
    ]
    if os.path.isdir("/content/drive/MyDrive"):
        for entry in os.scandir("/content/drive/MyDrive"):
            if entry.is_dir(follow_symlinks=False):
                for fn in ("task4_colab_pack.tar", "task4_colab_pack.zip"):
                    cand = os.path.join(entry.path, fn)
                    if os.path.isfile(cand):
                        direct_candidates.append(cand)

    pack_path = next((p for p in direct_candidates if os.path.isfile(p) and (p.endswith(".zip") or os.path.getsize(p) >= 700_000_000)), None)
    if pack_path is None:
        raise RuntimeError(parts_error or "Data pack not found! Right-click project/task4/data/task4_colab_pack_parts -> 'Upload to Colab'.")

    print("Extracting", pack_path, "->", DATA_DIR, "...")
    if pack_path.endswith(".zip"):
        with zipfile.ZipFile(pack_path, "r") as zf:
            zf.extractall(DATA_DIR)
    else:
        subprocess.check_call(["tar", "--exclude=._*", "--exclude=*/._*", "-xf", pack_path, "-C", DATA_DIR])
    if joined_from_parts and os.path.isfile(merged_tar):
        os.remove(merged_tar)

prepare_dataset_pack()
assert is_data_ready(DATA_DIR), "Extracted dataset incomplete!"
print("Ready! Train JPGs:", count_real_files(f"{DATA_DIR}/train"), "| Test JPGs:", count_real_files(f"{DATA_DIR}/test"))

Joining 37 verified chunks from /content/task4_colab_pack_parts -> /content/task4_colab_pack.tar ...
Verified pack SHA-256: d7316deeb95c78ff... (732.4 MiB)
Extracting /content/task4_colab_pack.tar -> /content/t4_data ...
Ready! Train JPGs: 31546 | Test JPGs: 25889


In [6]:
# Cell 3: Fast Spatial + GPU-Vectorized Nightmare Augmentations & ExtHard Validator
import io
import numpy as np
import torch
from PIL import Image, ImageFilter

def apply_nightmare_spatial_u8(pil_img: Image.Image, rng: np.random.Generator) -> np.ndarray:
    """Fast CPU spatial/blur/JPEG nightmare transform returning uint8 (photometric jitter runs on GPU)."""
    if rng.random() < 0.50:
        pil_img = pil_img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() < 0.55:
        ang = float(rng.uniform(-35.0, 35.0))
        pil_img = pil_img.rotate(ang, resample=Image.BILINEAR, fillcolor=(0, 0, 0))
    if rng.random() < 0.16:
        pil_img = pil_img.filter(ImageFilter.GaussianBlur(radius=float(rng.uniform(0.4, 1.3))))
    if rng.random() < 0.20:
        buf = io.BytesIO()
        pil_img.save(buf, format="JPEG", quality=int(rng.integers(28, 75)))
        buf.seek(0)
        pil_img = Image.open(buf).convert("RGB")
    return np.asarray(pil_img, dtype=np.uint8)

def apply_nightmare_photo_torch(x: torch.Tensor) -> torch.Tensor:
    """Vectorized GPU photometric nightmare corruptions on (N, 3, H, W) float32 in [0, 1]."""
    n = x.shape[0]
    if n == 0:
        return x
    dev = x.device
    # 1. Neon channel gain / color tint (p=0.55)
    m_tint = (torch.rand(n, device=dev) < 0.55).view(n, 1, 1, 1)
    gains = torch.empty((n, 3, 1, 1), device=dev, dtype=torch.float32).uniform_(0.30, 2.20)
    gains = gains / gains.mean(dim=1, keepdim=True).clamp(min=1e-4)
    x = torch.where(m_tint, (x * gains).clamp(0.0, 1.0), x)

    # 2. Exposure & contrast corruption (0.20 dark gamma / 0.20 overexpose / 0.14 washed)
    r_exp = torch.rand(n, device=dev)
    m_dark = (r_exp < 0.20).view(n, 1, 1, 1)
    m_over = ((r_exp >= 0.20) & (r_exp < 0.40)).view(n, 1, 1, 1)
    m_wash = ((r_exp >= 0.40) & (r_exp < 0.54)).view(n, 1, 1, 1)

    gamma = torch.empty((n, 1, 1, 1), device=dev).uniform_(1.8, 3.0)
    dark_sc = torch.empty((n, 1, 1, 1), device=dev).uniform_(0.32, 0.70)
    x_dark = x.clamp(min=1e-4).pow(gamma) * dark_sc

    ov_sc = torch.empty((n, 1, 1, 1), device=dev).uniform_(1.25, 2.10)
    ov_sh = torch.empty((n, 1, 1, 1), device=dev).uniform_(0.24, 0.48)
    x_over = (x * ov_sc + ov_sh).clamp(0.0, 1.0)

    w_sc = torch.empty((n, 1, 1, 1), device=dev).uniform_(0.30, 0.65)
    w_sh = torch.empty((n, 1, 1, 1), device=dev).uniform_(-0.15, 0.15)
    x_wash = (0.5 + (x - 0.5) * w_sc + w_sh).clamp(0.0, 1.0)

    x = torch.where(m_dark, x_dark, torch.where(m_over, x_over, torch.where(m_wash, x_wash, x)))

    # 3. Gaussian sensor noise (p=0.40)
    m_noise = (torch.rand(n, device=dev) < 0.40).view(n, 1, 1, 1)
    sig = torch.empty((n, 1, 1, 1), device=dev).uniform_(0.015, 0.075)
    x = torch.where(m_noise, (x + torch.randn_like(x) * sig).clamp(0.0, 1.0), x)
    return x

def apply_nightmare_aug(pil_img: Image.Image, rng: np.random.Generator) -> np.ndarray:
    """CPU nightmare helper retained for deterministic parity checks."""
    if rng.random() < 0.50:
        pil_img = pil_img.transpose(Image.FLIP_LEFT_RIGHT)
    if rng.random() < 0.55:
        ang = float(rng.uniform(-35.0, 35.0))
        pil_img = pil_img.rotate(ang, resample=Image.BILINEAR, fillcolor=(0, 0, 0))
    if rng.random() < 0.16:
        pil_img = pil_img.filter(ImageFilter.GaussianBlur(radius=float(rng.uniform(0.4, 1.3))))
    arr = np.asarray(pil_img, dtype=np.float32) * (1.0 / 255.0)
    if rng.random() < 0.55:
        gains = rng.uniform(0.30, 2.20, size=(1, 1, 3)).astype(np.float32)
        gains /= float(gains.mean())
        arr = np.clip(arr * gains, 0.0, 1.0)
    r_exp = rng.random()
    if r_exp < 0.20:
        arr = np.power(np.clip(arr, 1e-4, 1.0), float(rng.uniform(1.8, 3.0))) * float(rng.uniform(0.32, 0.70))
    elif r_exp < 0.40:
        arr = np.clip(arr * float(rng.uniform(1.25, 2.10)) + float(rng.uniform(0.24, 0.48)), 0.0, 1.0)
    elif r_exp < 0.54:
        arr = np.clip(0.5 + (arr - 0.5) * float(rng.uniform(0.30, 0.65)) + float(rng.uniform(-0.15, 0.15)), 0.0, 1.0)
    if rng.random() < 0.40:
        arr = arr + rng.normal(0.0, float(rng.uniform(0.015, 0.075)), size=arr.shape).astype(np.float32)
    arr_u8 = np.clip(arr * 255.0, 0.0, 255.0).astype(np.uint8)
    if rng.random() < 0.20:
        buf = io.BytesIO()
        Image.fromarray(arr_u8).save(buf, format="JPEG", quality=int(rng.integers(28, 75)))
        buf.seek(0)
        arr_u8 = np.asarray(Image.open(buf).convert("RGB"), dtype=np.uint8)
    return arr_u8

def make_deterministic_exthard(jpeg_bytes_list, indices: np.ndarray, seed: int = 123):
    """Generate deterministic ExtremeHard validation images for clean fold-level stress testing."""
    rng = np.random.default_rng(seed)
    out = []
    for k, idx in enumerate(indices):
        im = Image.open(io.BytesIO(jpeg_bytes_list[int(idx)])).convert("RGB")
        im = im.rotate(float(rng.uniform(-35.0, 35.0)), resample=Image.BILINEAR, fillcolor=(0, 0, 0))
        arr = np.asarray(im, dtype=np.float32) * (1.0 / 255.0)
        mode = k % 4
        if mode == 0:
            arr = np.power(np.clip(arr, 1e-4, 1.0), float(rng.uniform(1.8, 2.8))) * float(rng.uniform(0.35, 0.65))
        elif mode == 1:
            arr = np.clip(arr * float(rng.uniform(1.3, 2.0)) + float(rng.uniform(0.30, 0.50)), 0.0, 1.0)
        elif mode == 2:
            g = rng.uniform(0.35, 2.0, size=(1, 1, 3)).astype(np.float32)
            g /= float(g.max())
            arr = np.clip(np.power(arr, 0.85) * g * float(rng.uniform(0.8, 1.4)), 0.0, 1.0)
        else:
            arr = np.clip(0.5 + (arr - 0.5) * float(rng.uniform(0.4, 0.7)) + rng.uniform(-0.15, 0.15, (1, 1, 3)), 0.0, 1.0)
        arr = arr + rng.normal(0.0, float(rng.uniform(0.03, 0.07)), size=arr.shape).astype(np.float32)
        out.append(np.clip(arr * 255.0, 0.0, 255.0).astype(np.uint8))
    return out

print("Fast spatial + GPU-vectorized Nightmare & ExtHard helpers ready.")


Fast spatial + GPU-vectorized Nightmare & ExtHard helpers ready.


In [7]:
# Cell 4: Interior Gray-World Normalization (CPU Train View + Vectorized GPU Eval View)
import math
import numpy as np
import torch

def interior_gw_stretch_np(arr_u8: np.ndarray) -> np.ndarray:
    """Fast Gray-World + Percentile Contrast Stretch using central 60% interior pixels (CPU)."""
    h, w, _ = arr_u8.shape
    h0, h1 = int(h * 0.2), int(h * 0.8)
    w0, w1 = int(w * 0.2), int(w * 0.8)
    b = arr_u8.astype(np.float32) * (1.0 / 255.0)
    patch = b[h0:h1:2, w0:w1:2, :]
    ch_m = np.clip(patch.mean(axis=(0, 1), keepdims=True), 0.025, 1.0)
    b_gw = np.clip(b * (ch_m.mean(axis=2, keepdims=True) / ch_m), 0.0, 1.0)
    p_sub = b_gw[h0:h1:3, w0:w1:3, :].ravel()
    lo, hi = np.quantile(p_sub, (0.02, 0.98))
    b_st = np.clip((b_gw - float(lo)) / max(float(hi - lo), 0.10), 0.0, 1.0)
    m_int = float(np.clip(b_st[h0:h1:3, w0:w1:3, :].mean(), 0.05, 0.95))
    gamma = float(np.clip(math.log(0.45) / math.log(m_int), 0.50, 1.80))
    return np.clip(np.power(b_st, gamma), 0.0, 1.0).astype(np.float32)

def interior_gw_stretch_torch(b: torch.Tensor) -> torch.Tensor:
    """Vectorized GPU Gray-World + Percentile Contrast Stretch matching interior_gw_stretch_np."""
    _, _, h, w = b.shape
    h0, h1 = int(h * 0.2), int(h * 0.8)
    w0, w1 = int(w * 0.2), int(w * 0.8)
    patch = b[:, :, h0:h1:2, w0:w1:2]
    ch_m = patch.mean(dim=(2, 3), keepdim=True).clamp(0.025, 1.0)
    b_gw = (b * (ch_m.mean(dim=1, keepdim=True) / ch_m)).clamp(0.0, 1.0)
    p_sub = b_gw[:, :, h0:h1:3, w0:w1:3].reshape(b.shape[0], -1)
    q = torch.tensor([0.02, 0.98], device=b.device, dtype=b.dtype)
    qs = torch.quantile(p_sub, q, dim=1)
    lo = qs[0].view(-1, 1, 1, 1)
    hi = qs[1].view(-1, 1, 1, 1)
    b_st = ((b_gw - lo) / (hi - lo).clamp(min=0.10)).clamp(0.0, 1.0)
    m_int = b_st[:, :, h0:h1:3, w0:w1:3].mean(dim=(1, 2, 3)).clamp(0.05, 0.95)
    gamma = (math.log(0.45) / torch.log(m_int)).clamp(0.50, 1.80).view(-1, 1, 1, 1)
    return b_st.pow(gamma).clamp(0.0, 1.0)

print("CPU + Vectorized GPU Interior Gray-World normalizers ready.")

CPU + Vectorized GPU Interior Gray-World normalizers ready.


In [8]:
# Cell 5: JPEG RAM Preloader & Zero-Overhead uint8 Train/Eval Datasets (All Photometrics/Rotations Run on GPU)
import io, os
import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset

def load_jpeg_bytes(folder: str, filenames):
    """Preload compressed JPEG byte buffers into host RAM to eliminate disk I/O stalls."""
    buffers = []
    for fn in filenames:
        with open(os.path.join(folder, fn), "rb") as f:
            buffers.append(f.read())
    return buffers

class Task4TrainDataset(Dataset):
    """Fast worker dataset returning uint8 (3, H, W) + is_nightmare flag; GPU handles photometrics & GW norm."""

    def __init__(self, jpeg_bytes, indices, labels, img_hw, p_nightmare=0.50, p_norm_view=0.25, seed=42):
        self.jpeg_bytes = jpeg_bytes
        self.indices = indices
        self.labels = labels
        self.img_hw = img_hw
        self.p_nightmare = p_nightmare
        self.p_norm_view = p_norm_view
        self.seed = seed
        self._rng = None

    def _get_rng(self):
        if self._rng is None:
            info = torch.utils.data.get_worker_info()
            wid = info.id if info is not None else 0
            self._rng = np.random.default_rng(self.seed + wid * 7919)
        return self._rng

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, pos: int):
        rng = self._get_rng()
        idx = int(self.indices[pos])
        pil = Image.open(io.BytesIO(self.jpeg_bytes[idx])).convert("RGB")
        target_wh = (self.img_hw[1], self.img_hw[0])
        if pil.size != target_wh:
            pil = pil.resize(target_wh, resample=Image.BILINEAR)

        is_nm = bool(rng.random() < self.p_nightmare)
        if is_nm:
            arr_u8 = apply_nightmare_spatial_u8(pil, rng)
        else:
            if rng.random() < 0.50:
                pil = pil.transpose(Image.FLIP_LEFT_RIGHT)
            arr_u8 = np.asarray(pil, dtype=np.uint8)

        t_u8 = torch.from_numpy(np.array(arr_u8, dtype=np.uint8, copy=True)).permute(2, 0, 1).contiguous()
        return t_u8, is_nm, torch.tensor(int(self.labels[idx]), dtype=torch.long)

class Task4RawViewsDataset(Dataset):
    """Single-decode 1-frame uint8 dataset; flip, +-18deg rotation & Interior-GW run vectorized on GPU."""

    def __init__(self, source_items, indices, img_hw, need_rot: bool = False, is_bytes: bool = True):
        self.source_items = source_items
        self.indices = indices if indices is not None else np.arange(len(source_items))
        self.img_hw = img_hw
        self.need_rot = False  # Always perform +-18deg rotation on GPU via grid_sample
        self.is_bytes = is_bytes

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, pos: int):
        item = self.source_items[int(self.indices[pos])]
        pil = Image.open(io.BytesIO(item)).convert("RGB") if self.is_bytes else Image.fromarray(item)
        target_wh = (self.img_hw[1], self.img_hw[0])
        if pil.size != target_wh:
            pil = pil.resize(target_wh, resample=Image.BILINEAR)
        arr_u8 = np.array(pil, dtype=np.uint8, copy=True)
        return torch.from_numpy(arr_u8).permute(2, 0, 1).contiguous()

print("Zero-overhead uint8 train/eval datasets ready.")


Zero-overhead uint8 train/eval datasets ready.


In [9]:
# Cell 6: Model Specs, Smart Freeze, Batch/VRAM Selector & Optimizer Helpers
import math
from dataclasses import dataclass
from typing import Tuple
import timm
import torch
import torch.nn as nn

@dataclass(frozen=True)
class ModelSpec:
    """Configuration for a corruption-hardened Task 4 vision backbone."""
    key: str
    timm_name: str
    img_hw: Tuple[int, int]
    epochs: int
    base_lr: float
    backbone_lr_mult: float
    batch_a100: int
    batch_l4: int
    batch_t4: int
    eval_batch_t4: int
    grad_ckpt: bool
    use_img_size_arg: bool
    channels_last_ampere: bool
    default_freeze_vit_blocks: int

MODEL_SPECS = {
    "cnxv3": ModelSpec("cnxv3", "convnext_base.dinov3_lvd1689m", (192, 384), 6, 3.0e-4, 0.25, 96, 64, 48, 160, False, False, True, 0),
    "cnxv2b": ModelSpec("cnxv2b", "convnextv2_base.fcmae_ft_in22k_in1k", (192, 384), 6, 3.0e-4, 0.25, 96, 64, 48, 160, False, False, True, 0),
    "eva02l": ModelSpec("eva02l", "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k", (196, 392), 4, 2.0e-4, 0.22, 48, 32, 16, 96, True, True, False, 6),
    "dinov3l": ModelSpec("dinov3l", "vit_large_patch16_dinov3.lvd1689m", (192, 384), 5, 2.0e-4, 0.22, 64, 32, 16, 128, True, True, False, 6),
}

def apply_t4_freeze(model: nn.Module, spec: ModelSpec, freeze_vit_blocks: int = 6):
    """Freeze only lowest ViT blocks (never patch_embed/tokens or ConvNeXt stem/stage0)."""
    total_params = sum(p.numel() for p in model.parameters())
    if spec.default_freeze_vit_blocks > 0 and freeze_vit_blocks > 0:
        for name, param in model.named_parameters():
            if name.startswith("blocks."):
                parts = name.split(".")
                if len(parts) > 1 and parts[1].isdigit() and int(parts[1]) < freeze_vit_blocks:
                    param.requires_grad_(False)
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return trainable_params, total_params

def build_model(spec: ModelSpec, pretrained: bool = True, freeze_vit_blocks: int = 6):
    """Instantiate timm model with exact 2:1 aspect-ratio img_size, smart freeze, and RGB mean/std."""
    kwargs = {"pretrained": pretrained, "num_classes": 5}
    if spec.use_img_size_arg:
        kwargs["img_size"] = spec.img_hw
    model = timm.create_model(spec.timm_name, **kwargs)
    if spec.grad_ckpt and hasattr(model, "set_grad_checkpointing"):
        model.set_grad_checkpointing(True)
    param_counts = apply_t4_freeze(model, spec, freeze_vit_blocks=freeze_vit_blocks)
    cfg = timm.data.resolve_model_data_config(model)
    return model, tuple(cfg["mean"]), tuple(cfg["std"]), param_counts

def pick_batch_and_accum(spec: ModelSpec, device: torch.device, target_eff_bs: int = 48):
    """Select train batch size, grad accumulation, and eval batch size from detected GPU VRAM."""
    if device.type != "cuda":
        return 8, 1, 16
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    if vram_gb >= 36.0:
        bs, eval_bs = spec.batch_a100, 96
    elif vram_gb >= 21.0:
        bs, eval_bs = spec.batch_l4, 64
    else:
        bs, eval_bs = spec.batch_t4, spec.eval_batch_t4
    return bs, max(1, math.ceil(target_eff_bs / bs)), eval_bs

def make_param_groups(model: nn.Module, base_lr: float, bb_mult: float, wd: float = 2e-2):
    """Separate classifier head vs backbone parameters and exclude 1D norms/biases from weight decay."""
    hd_d, hd_n, bb_d, bb_n = [], [], [], []
    for name, p in model.named_parameters():
        if not p.requires_grad:
            continue
        is_h = name.startswith("head") or name.startswith("fc") or ".head." in name
        no_w = p.ndim <= 1 or name.endswith(".bias") or "norm" in name.lower()
        target = (hd_n if no_w else hd_d) if is_h else (bb_n if no_w else bb_d)
        target.append(p)
    groups = []
    for lst, lr_v, wd_v in [(hd_d, base_lr, wd), (hd_n, base_lr, 0.0), (bb_d, base_lr * bb_mult, wd), (bb_n, base_lr * bb_mult, 0.0)]:
        if lst:
            groups.append({"params": lst, "lr": lr_v, "weight_decay": wd_v})
    return groups

def update_ema(ema_state, model: nn.Module, decay: float = 0.996):
    """Update exponential moving average shadow weights on device."""
    with torch.no_grad():
        for k, v in model.state_dict().items():
            if k in ema_state and v.dtype.is_floating_point:
                ema_state[k].mul_(decay).add_(v.detach(), alpha=1.0 - decay)
            elif k not in ema_state:
                ema_state[k] = v.detach().clone()

print("Model specs & optimizer helpers ready:", list(MODEL_SPECS.keys()))

Model specs & optimizer helpers ready: ['cnxv3', 'cnxv2b', 'eva02l', 'dinov3l']


In [10]:
# Cell 7: CUDA / T4 Runtime Engine Preflight Probe (Prevents cuDNN FIND Crashes)
import torch
import torch.nn as nn
import torch.nn.functional as F

def probe_runtime_engine(model: nn.Module, spec: ModelSpec, device: torch.device, amp_dtype, use_scaler: bool):
    """Test forward+backward and odd-batch eval before Epoch 0; falls back cleanly if cuDNN FIND fails."""
    h, w = spec.img_hw
    if device.type != "cuda":
        xb = torch.randn(2, 3, h, w, device=device)
        yb = torch.tensor([0, 1], device=device, dtype=torch.long)
        model.train()
        loss = F.cross_entropy(model(xb).float(), yb)
        loss.backward()
        assert any(p.grad is not None for p in model.parameters() if p.requires_grad), "No trainable grads after backward"
        model.zero_grad(set_to_none=True)
        model.eval()
        with torch.no_grad():
            _ = model(xb)
        return False, f"{device.type}-fp32"

    cc_major, cc_minor = torch.cuda.get_device_capability(0)
    candidates = []
    if cc_major >= 8:
        if spec.channels_last_ampere:
            candidates.append((True, True, True, f"sm{cc_major}{cc_minor}-nhwc-cudnn"))
        candidates.append((False, True, True, f"sm{cc_major}{cc_minor}-nchw-cudnn"))
        candidates.append((False, True, False, f"sm{cc_major}{cc_minor}-nchw-cudnn-safe"))
        candidates.append((False, False, False, f"sm{cc_major}{cc_minor}-nchw-nocudnn"))
    else:
        candidates.append((False, True, False, f"sm{cc_major}{cc_minor}-nchw-cudnn-safe"))
        candidates.append((False, False, False, f"sm{cc_major}{cc_minor}-nchw-nocudnn"))

    last_exc = None
    for use_cl, use_cudnn, use_bench, tag in candidates:
        try:
            torch.backends.cudnn.enabled = use_cudnn
            torch.backends.cudnn.benchmark = use_bench
            model.to(memory_format=(torch.channels_last if use_cl else torch.contiguous_format))
            model.zero_grad(set_to_none=True)

            probe_scaler = torch.amp.GradScaler("cuda", init_scale=1024.0, enabled=use_scaler)
            xb = torch.randn(2, 3, h, w, device=device)
            if use_cl:
                xb = xb.to(memory_format=torch.channels_last)
            yb = torch.tensor([0, 1], device=device, dtype=torch.long)

            model.train()
            with torch.amp.autocast(device.type, dtype=amp_dtype) if amp_dtype is not None else torch.enable_grad():
                logits = model(xb)
                assert torch.isfinite(logits.float()).all(), "Non-finite logits in preflight probe"
                loss = F.cross_entropy(logits.float(), yb)
            if use_scaler:
                probe_scaler.scale(loss).backward()
            else:
                loss.backward()
            has_grad = any(p.grad is not None for p in model.parameters() if p.requires_grad)
            assert has_grad, "Gradients missing in preflight probe"
            model.zero_grad(set_to_none=True)

            model.eval()
            xb_odd = torch.randn(3, 3, h, w, device=device)
            if use_cl:
                xb_odd = xb_odd.to(memory_format=torch.channels_last)
            with torch.no_grad(), (torch.amp.autocast(device.type, dtype=amp_dtype) if amp_dtype is not None else torch.no_grad()):
                _ = model(xb_odd)
            torch.cuda.synchronize()
            return use_cl, tag
        except Exception as exc:
            last_exc = exc
            model.zero_grad(set_to_none=True)
            torch.cuda.empty_cache()

    raise RuntimeError(f"All runtime engine candidates failed for {spec.key}: {last_exc}") from last_exc

print("Runtime engine preflight probe ready.")

Runtime engine preflight probe ready.


In [11]:
# Cell 8: Vectorized GPU 6-View TTA Inference, Geometric Pooling & Epoch/Fold Checkpointers
import math, os, shutil
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

def gpu_rotate_batch(x: torch.Tensor, angle_deg: float) -> torch.Tensor:
    """Aspect-preserving GPU bilinear rotation matching PIL.Image.rotate(angle_deg, BILINEAR)."""
    b, _, h, w = x.shape
    theta = math.radians(angle_deg)
    cos_t, sin_t = math.cos(theta), math.sin(theta)
    mat = torch.tensor(
        [[cos_t, -(h / w) * sin_t, 0.0],
         [(w / h) * sin_t, cos_t, 0.0]],
        device=x.device, dtype=x.dtype,
    ).unsqueeze(0).expand(b, -1, -1)
    grid = F.affine_grid(mat, size=x.size(), align_corners=False)
    return F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)

def _build_gpu_tta_batch(raw_f: torch.Tensor, views):
    """Expand raw (B, 3, H, W) [or (B, R, 3, H, W)] float32 [0,1] frames into (B, V, 3, H, W) on GPU."""
    x_orig = raw_f[:, 0] if raw_f.ndim == 5 else raw_f
    x_flip = torch.flip(x_orig, dims=[-1]) if any(v in ("flip", "gw_flip") for v in views) else None
    need_gw = any(v in ("gw", "gw_flip") for v in views)
    x_gw = interior_gw_stretch_torch(x_orig) if "gw" in views else None
    x_gw_flip = interior_gw_stretch_torch(x_flip) if "gw_flip" in views else None
    built = []
    for v in views:
        if v == "orig":
            built.append(x_orig)
        elif v == "flip":
            built.append(x_flip)
        elif v == "gw":
            built.append(x_gw)
        elif v == "gw_flip":
            built.append(x_gw_flip)
        elif v == "rot_p18_gw":
            x_r = raw_f[:, 1] if (raw_f.ndim == 5 and raw_f.shape[1] >= 2) else gpu_rotate_batch(x_orig, 18.0)
            built.append(interior_gw_stretch_torch(x_r))
        elif v == "rot_m18_gw":
            x_r = raw_f[:, 2] if (raw_f.ndim == 5 and raw_f.shape[1] >= 3) else gpu_rotate_batch(x_orig, -18.0)
            built.append(interior_gw_stretch_torch(x_r))
        else:
            raise ValueError(f"Unknown TTA view: {v}")
    return torch.stack(built, dim=1)

@torch.no_grad()
def predict_views(model, source_items, indices, spec: ModelSpec, mean_t, std_t, views, eval_bs: int, device, amp_dtype, use_channels_last=False, is_bytes=True, num_workers=2):
    """Run 1-frame uint8 decode + GPU rotation/GW 6-view TTA with grad-checkpointing disabled and OOM backoff."""
    model.eval()
    was_gc = bool(getattr(model, "grad_checkpointing", False))
    if was_gc and hasattr(model, "set_grad_checkpointing"):
        model.set_grad_checkpointing(False)

    n = len(indices) if indices is not None else len(source_items)
    v_cnt = len(views)
    dl_bs = max(64, min(192, eval_bs)) if v_cnt == 1 else max(32, min(96, eval_bs))
    ds = Task4RawViewsDataset(source_items, indices, spec.img_hw, need_rot=False, is_bytes=is_bytes)
    dl_kw = {"batch_size": dl_bs, "shuffle": False, "num_workers": num_workers, "pin_memory": (device.type == "cuda")}
    if num_workers > 0:
        dl_kw["prefetch_factor"] = 4
    dl = DataLoader(ds, **dl_kw)

    cur_bs = int(eval_bs)
    all_probs = []
    try:
        for batch_u8 in dl:
            bsz = batch_u8.shape[0]
            raw_f = batch_u8.to(device, non_blocking=True).float() * (1.0 / 255.0)
            views_t = _build_gpu_tta_batch(raw_f, views)
            flat = views_t.view(bsz * v_cnt, 3, views_t.shape[3], views_t.shape[4])
            out_chunks = []
            s = 0
            while s < len(flat):
                xb = flat[s : s + cur_bs]
                try:
                    if use_channels_last and device.type == "cuda":
                        xb = xb.to(memory_format=torch.channels_last)
                    xb = (xb - mean_t) / std_t
                    with torch.amp.autocast(device.type, dtype=amp_dtype) if amp_dtype is not None else torch.no_grad():
                        logits = model(xb)
                    out_chunks.append(F.softmax(logits.float(), dim=1).cpu().numpy())
                    s += len(xb)
                except RuntimeError as exc:
                    if "out of memory" not in str(exc).lower():
                        raise
                    if device.type == "cuda":
                        torch.cuda.empty_cache()
                    if cur_bs <= 16:
                        raise
                    cur_bs = max(16, cur_bs // 2)
                    print(f"[{spec.key}] Eval OOM backoff -> eval_bs={cur_bs}", flush=True)
            all_probs.append(np.concatenate(out_chunks, axis=0).reshape(bsz, v_cnt, 5))
    finally:
        if was_gc and hasattr(model, "set_grad_checkpointing"):
            model.set_grad_checkpointing(True)

    merged = np.concatenate(all_probs, axis=0) if all_probs else np.zeros((0, v_cnt, 5), dtype=np.float32)
    assert len(merged) == n
    return {v: merged[:, i, :].copy() for i, v in enumerate(views)}

def combine_tta_views(vdict, mode: str) -> np.ndarray:
    """Combine view probabilities in geometric log-space."""
    if mode == "orig2":
        keys, ws = ["orig", "flip"], [0.55, 0.45]
    elif mode == "weighted6":
        keys, ws = ["orig", "flip", "gw", "gw_flip", "rot_p18_gw", "rot_m18_gw"], [0.28, 0.20, 0.20, 0.12, 0.10, 0.10]
    else:
        raise ValueError(f"Unknown combine mode: {mode}")
    lp = sum(w * np.log(np.clip(vdict[k], 1e-7, 1.0)) for k, w in zip(keys, ws))
    lp -= lp.max(axis=1, keepdims=True)
    ep = np.exp(lp)
    return (ep / ep.sum(axis=1, keepdims=True)).astype(np.float32)

def save_csv_preds(out_path: str, df_meta: pd.DataFrame, probs: np.ndarray, folds_col=None):
    """Write predictions matching project/task4/predictions/model-level CSV schema."""
    out = pd.DataFrame({"id": df_meta["id"].values, "file_name": df_meta["file_name"].values})
    if "label" in df_meta.columns and df_meta["label"].notna().all():
        out["label"] = df_meta["label"].astype(int).values
    if folds_col is not None:
        out["fold"] = folds_col.astype(int)
    for c in range(5):
        out[f"prob_{c}"] = np.round(probs[:, c].astype(np.float64), 7)
    out.to_csv(out_path, index=False)

def atomic_savez_compressed(path: str, **arrays):
    """Atomically write compressed .npz file so disconnects never corrupt checkpoints."""
    tmp_path = f"{path}.tmp.{os.getpid()}.npz"
    np.savez_compressed(tmp_path, **arrays)
    os.replace(tmp_path, path)

def load_fold_npz(path: str):
    """Safely load completed fold checkpoint if present and intact."""
    req = ("va_orig2", "va_w6", "ext_w6", "te_orig2", "te_w6")
    if not os.path.isfile(path):
        return None
    try:
        with np.load(path) as z:
            if not all(k in z for k in req):
                return None
            return {k: z[k].copy() for k in req}
    except Exception:
        return None

def resolve_epoch_ckpt_paths(cache_dir: str, model_key: str, k: int):
    drive_ckpt = os.path.join(cache_dir, f"fold{k}_ckpt.pt")
    if os.path.isdir("/content"):
        local_dir = os.path.join("/content/t4_hard_ckpt_tmp", f"folds_{model_key}")
        os.makedirs(local_dir, exist_ok=True)
        local_ckpt = os.path.join(local_dir, f"fold{k}_ckpt.pt")
    else:
        local_ckpt = drive_ckpt
    return drive_ckpt, local_ckpt

def save_epoch_ckpt(cache_dir: str, spec: ModelSpec, k: int, ep: int, epochs: int, freeze_vit_blocks: int, model, ema_state, opt, sched, scaler) -> str:
    """Save only trainable/buffer tensors + optimizer/scheduler state atomically for zero-loss resume."""
    drive_ckpt, local_ckpt = resolve_epoch_ckpt_paths(cache_dir, spec.key, k)
    trainable_keys = {n for n, p in model.named_parameters() if p.requires_grad}
    buf_keys = {n for n, _ in model.named_buffers()}
    keep_keys = trainable_keys | buf_keys

    payload = {
        "spec_key": spec.key,
        "img_hw": tuple(spec.img_hw),
        "epochs": int(epochs),
        "freeze_vit_blocks": int(freeze_vit_blocks),
        "epoch": int(ep),
        "trainable_model": {sk: sv.detach().cpu() for sk, sv in model.state_dict().items() if sk in keep_keys},
        "ema_state": {ek: ev.detach().cpu() for ek, ev in ema_state.items() if ek in keep_keys},
        "opt": opt.state_dict(),
        "sched": sched.state_dict(),
        "scaler": scaler.state_dict(),
    }

    target = drive_ckpt
    try:
        if drive_ckpt != local_ckpt and shutil.disk_usage(cache_dir).free < 3_200_000_000:
            target = local_ckpt
    except OSError:
        target = local_ckpt

    tmp_path = f"{target}.tmp.{os.getpid()}"
    try:
        torch.save(payload, tmp_path)
        os.replace(tmp_path, target)
    except OSError:
        if os.path.isfile(tmp_path):
            os.remove(tmp_path)
        if target != local_ckpt:
            tmp_fb = f"{local_ckpt}.tmp.{os.getpid()}"
            torch.save(payload, tmp_fb)
            os.replace(tmp_fb, local_ckpt)
            target = local_ckpt
        else:
            raise
    return target

def load_epoch_ckpt(cache_dir: str, spec: ModelSpec, k: int, epochs: int, freeze_vit_blocks: int):
    drive_ckpt, local_ckpt = resolve_epoch_ckpt_paths(cache_dir, spec.key, k)
    cands = [drive_ckpt] if drive_ckpt == local_ckpt else [drive_ckpt, local_ckpt]
    best_payload = None
    for cp in cands:
        if not os.path.isfile(cp):
            continue
        try:
            pl = torch.load(cp, map_location="cpu", weights_only=False)
            if (
                pl.get("spec_key") == spec.key
                and tuple(pl.get("img_hw", ())) == tuple(spec.img_hw)
                and int(pl.get("epochs", -1)) == int(epochs)
                and int(pl.get("freeze_vit_blocks", -1)) == int(freeze_vit_blocks)
            ):
                if best_payload is None or int(pl.get("epoch", -1)) > int(best_payload.get("epoch", -1)):
                    best_payload = pl
        except Exception:
            continue
    return best_payload

def remove_epoch_ckpt(cache_dir: str, model_key: str, k: int):
    drive_ckpt, local_ckpt = resolve_epoch_ckpt_paths(cache_dir, model_key, k)
    for cp in {drive_ckpt, local_ckpt}:
        if os.path.isfile(cp):
            os.remove(cp)

print("GPU TTA inference & epoch/fold atomic checkpoint helpers ready.")

GPU TTA inference & epoch/fold atomic checkpoint helpers ready.


In [12]:
# Cell 9: Single-Fold Trainer (Epoch Auto-Resume + EMA + GPU 6-View OOF/ExtHard/Test)
import gc, math, os, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score
from torch.utils.data import DataLoader

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def train_single_fold(
    k: int,
    spec: ModelSpec,
    tr_idx,
    va_idx,
    tr_bytes,
    te_bytes,
    y,
    bs: int,
    accum: int,
    eval_bs: int,
    epochs: int,
    eff_workers: int,
    views,
    cache_dir: str,
    resume: bool,
    device: torch.device,
    amp_dtype,
    use_scaler: bool,
    pretrained: bool,
    freeze_vit_blocks: int,
    p_nightmare: float,
    p_norm_view: float,
    seed: int,
):
    """Train one SGKF fold with epoch checkpoints, EMA tracking, and GPU 6-view OOF/ExtHard/Test."""
    t0 = time.time()
    seed_everything(seed + k * 101)
    model, mean_tup, std_tup, (n_train, n_tot) = build_model(
        spec,
        pretrained=pretrained,
        freeze_vit_blocks=freeze_vit_blocks,
    )
    model = model.to(device)
    use_cl, runtime_tag = probe_runtime_engine(model, spec, device, amp_dtype, use_scaler)
    seed_everything(seed + k * 101)

    mean_t = torch.tensor(mean_tup, device=device, dtype=torch.float32).view(1, 3, 1, 1)
    std_t = torch.tensor(std_tup, device=device, dtype=torch.float32).view(1, 3, 1, 1)

    pi_tr = np.bincount(y[tr_idx], minlength=5) / max(1, len(tr_idx))
    logit_adj = torch.tensor(0.20 * np.log(np.clip(pi_tr, 1e-4, 1.0)), device=device, dtype=torch.float32)

    tr_ds = Task4TrainDataset(tr_bytes, tr_idx, y, spec.img_hw, p_nightmare=p_nightmare, p_norm_view=p_norm_view, seed=seed + k * 1000)
    tr_dl_kw = {
        "batch_size": bs,
        "shuffle": True,
        "drop_last": len(tr_ds) >= bs,
        "num_workers": eff_workers,
        "pin_memory": (device.type == "cuda"),
        "persistent_workers": (eff_workers > 0),
    }
    if eff_workers > 0:
        tr_dl_kw["prefetch_factor"] = 4
    tr_dl = DataLoader(tr_ds, **tr_dl_kw)

    opt = torch.optim.AdamW(make_param_groups(model, spec.base_lr, spec.backbone_lr_mult))
    total_steps = max(1, math.ceil(len(tr_dl) / accum)) * epochs
    warmup_steps = max(1, int(0.08 * total_steps))
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt,
        lambda s: (s + 1) / warmup_steps if s < warmup_steps else 0.05 + 0.95 * 0.5 * (1.0 + math.cos(math.pi * (s - warmup_steps) / max(1, total_steps - warmup_steps))),
    )
    scaler = torch.amp.GradScaler("cuda", enabled=use_scaler)
    ema_state = {}

    start_ep = 0
    ckpt = load_epoch_ckpt(cache_dir, spec, k, epochs, freeze_vit_blocks) if resume else None
    if ckpt is not None:
        model.load_state_dict(ckpt["trainable_model"], strict=False)
        for ek, ev in ckpt["ema_state"].items():
            ema_state[ek] = ev.to(device=device)
        opt.load_state_dict(ckpt["opt"])
        sched.load_state_dict(ckpt["sched"])
        if ckpt.get("scaler"):
            scaler.load_state_dict(ckpt["scaler"])
        start_ep = int(ckpt["epoch"]) + 1
        print(
            f"[{spec.key}] Fold {k} resumed mid-fold at Ep {start_ep}/{epochs} | "
            f"engine={runtime_tag} | amp={amp_dtype} | trainable={n_train/1e6:.1f}M/{n_tot/1e6:.1f}M",
            flush=True,
        )
    else:
        print(
            f"[{spec.key}] Fold {k} started | engine={runtime_tag} | amp={amp_dtype} | "
            f"trainable={n_train/1e6:.1f}M/{n_tot/1e6:.1f}M",
            flush=True,
        )

    for ep in range(start_ep, epochs):
        seed_everything(seed + k * 101 + ep * 17)
        model.train()
        opt.zero_grad(set_to_none=True)
        ep_loss, n_seen = 0.0, 0
        for step_i, (xb_u8, is_nm, yb) in enumerate(tr_dl):
            xb = xb_u8.to(device, non_blocking=True).float() * (1.0 / 255.0)
            nm_mask = is_nm.to(device=device, dtype=torch.bool, non_blocking=True)
            if nm_mask.any():
                xb[nm_mask] = apply_nightmare_photo_torch(xb[nm_mask])
            if p_norm_view > 0.0:
                gw_mask = torch.rand(xb.shape[0], device=device) < p_norm_view
                if gw_mask.any():
                    xb[gw_mask] = interior_gw_stretch_torch(xb[gw_mask])
            if use_cl and device.type == "cuda":
                xb = xb.to(memory_format=torch.channels_last)
            yb = yb.to(device, non_blocking=True)
            xb = (xb - mean_t) / std_t

            with torch.amp.autocast(device.type, dtype=amp_dtype) if amp_dtype is not None else torch.enable_grad():
                logits = model(xb)
                loss = F.cross_entropy(logits.float() + logit_adj, yb, label_smoothing=0.03)
                loss_scaled = loss / accum

            if use_scaler:
                scaler.scale(loss_scaled).backward()
            else:
                loss_scaled.backward()

            if ((step_i + 1) % accum == 0) or ((step_i + 1) == len(tr_dl)):
                if use_scaler:
                    scaler.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), 2.0)
                if use_scaler:
                    prev_scale = scaler.get_scale()
                    scaler.step(opt)
                    scaler.update()
                    if scaler.get_scale() >= prev_scale:
                        sched.step()
                else:
                    opt.step()
                    sched.step()
                opt.zero_grad(set_to_none=True)
                update_ema(ema_state, model, decay=0.996 if ep >= 1 else 0.98)

            ep_loss += float(loss.item()) * len(yb)
            n_seen += len(yb)

        val_q = predict_views(model, tr_bytes, va_idx, spec, mean_t, std_t, ["orig"], eval_bs, device, amp_dtype, use_cl, True, eff_workers)["orig"]
        vf1 = f1_score(y[va_idx], val_q.argmax(1), average="macro")
        ckpt_saved = save_epoch_ckpt(cache_dir, spec, k, ep, epochs, freeze_vit_blocks, model, ema_state, opt, sched, scaler)
        ckpt_loc = ("Drive:" if "/drive/" in ckpt_saved else "Local:") + os.path.basename(ckpt_saved)
        print(
            f"[{spec.key}] Fold {k} Ep {ep+1}/{epochs} loss={ep_loss/max(1, n_seen):.4f} "
            f"ValCleanF1={vf1:.5f} | saved ckpt ({ckpt_loc}) ({time.time()-t0:.1f}s)",
            flush=True,
        )

    model.load_state_dict(ema_state, strict=False)
    t_ev = time.time()
    va_views = predict_views(model, tr_bytes, va_idx, spec, mean_t, std_t, views, eval_bs, device, amp_dtype, use_cl, True, eff_workers)
    ext_imgs = make_deterministic_exthard(tr_bytes, va_idx, seed=123 + k)
    ext_views = predict_views(model, ext_imgs, None, spec, mean_t, std_t, views, eval_bs, device, amp_dtype, use_cl, False, eff_workers)
    te_views = predict_views(model, te_bytes, None, spec, mean_t, std_t, views, eval_bs, device, amp_dtype, use_cl, True, eff_workers)

    fold_out = {
        "va_orig2": combine_tta_views(va_views, "orig2"),
        "va_w6": combine_tta_views(va_views, "weighted6"),
        "ext_w6": combine_tta_views(ext_views, "weighted6"),
        "te_orig2": combine_tta_views(te_views, "orig2"),
        "te_w6": combine_tta_views(te_views, "weighted6"),
    }
    f_val = f1_score(y[va_idx], fold_out["va_w6"].argmax(1), average="macro")
    f_ext = f1_score(y[va_idx], fold_out["ext_w6"].argmax(1), average="macro")
    print(
        f"[{spec.key}] DONE Fold {k} in {time.time()-t0:.1f}s (eval={time.time()-t_ev:.1f}s) | "
        f"Val6TTA F1={f_val:.5f} | ExtHard F1={f_ext:.5f}",
        flush=True,
    )

    del model, opt, sched, scaler, ema_state, tr_dl, tr_ds, ext_imgs, va_views, ext_views, te_views
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    return fold_out

print("Single-fold epoch-resumable trainer ready.")

Single-fold epoch-resumable trainer ready.


In [13]:
# Cell 10: 5-Fold SGKF Orchestrator & Per-Model ZIP Packager
import os, shutil, zipfile
import numpy as np
from sklearn.metrics import f1_score, log_loss

def package_model_zip(out_dir: str, model_key: str) -> str:
    """Package completed CSV/NPZ predictions for one backbone into a downloadable ZIP."""
    model_files = [
        f"task4-{model_key}-hard6-oof.csv",
        f"task4-{model_key}-hard6-test-predictions.csv",
        f"task4-{model_key}-orig2-oof.csv",
        f"task4-{model_key}-orig2-test-predictions.csv",
        f"task4-{model_key}-sgkf-hard-bundle.npz",
    ]
    mzip_out = os.path.join(out_dir, f"t4_hard_{model_key}.zip")
    with zipfile.ZipFile(mzip_out, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for fn in model_files:
            fp = os.path.join(out_dir, fn)
            if os.path.isfile(fp):
                zf.write(fp, fn)
    if os.path.isdir("/content") and os.path.abspath(mzip_out) != f"/content/t4_hard_{model_key}.zip":
        shutil.copy2(mzip_out, f"/content/t4_hard_{model_key}.zip")
    return mzip_out

def train_one_model(
    spec: ModelSpec,
    tr_bytes,
    te_bytes,
    y,
    folds,
    tr_df,
    te_df,
    out_dir: str,
    device,
    pretrained=True,
    epochs_override=None,
    max_folds=5,
    num_workers=2,
    resume=True,
    freeze_vit_blocks=None,
    p_nightmare=0.50,
    p_norm_view=0.25,
    seed=42,
):
    """Run or resume all 5 SGKF folds + epoch ckpts for one backbone and export CSV + NPZ + ZIP."""
    os.makedirs(out_dir, exist_ok=True)
    cache_dir = os.path.join(out_dir, f"folds_{spec.key}")
    os.makedirs(cache_dir, exist_ok=True)

    amp_dtype, use_scaler = select_amp_dtype(device)
    bs, accum, eval_bs = pick_batch_and_accum(spec, device)
    epochs = epochs_override if epochs_override is not None else spec.epochs
    eff_freeze = spec.default_freeze_vit_blocks if freeze_vit_blocks is None else freeze_vit_blocks
    eff_workers = min(num_workers, max(1, os.cpu_count() or num_workers)) if num_workers > 0 else 0
    views = ["orig", "flip", "gw", "gw_flip", "rot_p18_gw", "rot_m18_gw"]
    print(
        f"[{spec.key}] Config: img_hw={spec.img_hw} | bs={bs} (accum={accum}, eval_bs={eval_bs}) | "
        f"epochs={epochs} | amp={amp_dtype} | freeze_vit_blocks={eff_freeze} | workers={eff_workers}"
    )

    n_tr, n_te = len(tr_df), len(te_df)
    oof_orig2 = np.zeros((n_tr, 5), dtype=np.float32)
    oof_w6 = np.zeros((n_tr, 5), dtype=np.float32)
    ext_w6 = np.zeros((n_tr, 5), dtype=np.float32)
    te_orig2 = np.zeros((n_te, 5), dtype=np.float32)
    te_w6 = np.zeros((n_te, 5), dtype=np.float32)
    folds_done = 0

    for k in range(min(5, max_folds)):
        fold_npz = os.path.join(cache_dir, f"fold{k}.npz")
        va_idx, tr_idx = np.where(folds == k)[0], np.where(folds != k)[0]

        cached = load_fold_npz(fold_npz) if resume else None
        if cached is not None:
            remove_epoch_ckpt(cache_dir, spec.key, k)
            oof_orig2[va_idx], oof_w6[va_idx], ext_w6[va_idx] = cached["va_orig2"], cached["va_w6"], cached["ext_w6"]
            te_orig2 += cached["te_orig2"]
            te_w6 += cached["te_w6"]
            folds_done += 1
            print(f"[{spec.key}] Fold {k} resumed -> ValF1={f1_score(y[va_idx], oof_w6[va_idx].argmax(1), average='macro'):.5f} | ExtHardF1={f1_score(y[va_idx], ext_w6[va_idx].argmax(1), average='macro'):.5f}")
            continue

        fold_res = train_single_fold(
            k=k, spec=spec, tr_idx=tr_idx, va_idx=va_idx, tr_bytes=tr_bytes, te_bytes=te_bytes, y=y,
            bs=bs, accum=accum, eval_bs=eval_bs, epochs=epochs, eff_workers=eff_workers, views=views,
            cache_dir=cache_dir, resume=resume, device=device, amp_dtype=amp_dtype, use_scaler=use_scaler,
            pretrained=pretrained, freeze_vit_blocks=eff_freeze, p_nightmare=p_nightmare, p_norm_view=p_norm_view, seed=seed,
        )
        atomic_savez_compressed(fold_npz, **fold_res)
        remove_epoch_ckpt(cache_dir, spec.key, k)
        oof_orig2[va_idx], oof_w6[va_idx], ext_w6[va_idx] = fold_res["va_orig2"], fold_res["va_w6"], fold_res["ext_w6"]
        te_orig2 += fold_res["te_orig2"]
        te_w6 += fold_res["te_w6"]
        folds_done += 1

    if folds_done == 5:
        te_orig2 /= 5.0
        te_w6 /= 5.0
        f_all = f1_score(y, oof_w6.argmax(1), average="macro")
        f_o2 = f1_score(y, oof_orig2.argmax(1), average="macro")
        f_ext = f1_score(y, ext_w6.argmax(1), average="macro")
        ll_all = log_loss(y, np.clip(oof_w6, 1e-7, 1.0), labels=[0, 1, 2, 3, 4])
        print(
            f"\n=== [{spec.key}] 5-FOLD COMPLETE | OOF_W6 F1={f_all:.5f} LL={ll_all:.5f} | "
            f"Orig2 F1={f_o2:.5f} | ExtHard F1={f_ext:.5f} | Combo={0.5*(f_all+f_ext):.5f} ===\n",
            flush=True,
        )
        save_csv_preds(os.path.join(out_dir, f"task4-{spec.key}-hard6-oof.csv"), tr_df, oof_w6, folds)
        save_csv_preds(os.path.join(out_dir, f"task4-{spec.key}-hard6-test-predictions.csv"), te_df, te_w6)
        save_csv_preds(os.path.join(out_dir, f"task4-{spec.key}-orig2-oof.csv"), tr_df, oof_orig2, folds)
        save_csv_preds(os.path.join(out_dir, f"task4-{spec.key}-orig2-test-predictions.csv"), te_df, te_orig2)
        atomic_savez_compressed(
            os.path.join(out_dir, f"task4-{spec.key}-sgkf-hard-bundle.npz"),
            folds=folds, oof_orig2=oof_orig2, oof_w6=oof_w6, ext_w6=ext_w6, te_orig2=te_orig2, te_w6=te_w6,
        )
        mzip = package_model_zip(out_dir, spec.key)
        print(f"[READY] Saved 5-fold {spec.key} zip -> {mzip}", flush=True)

print("5-fold orchestrator ready.")

5-fold orchestrator ready.


In [14]:
# Cell 11: Preload Manifest + JPEG Buffers & Run Selected Backbones Live
import os, time
import numpy as np
import pandas as pd

tr_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
te_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
cmap = {"7-11k": 0, "7-11K": 0, "fiveN": 1, "freefried": 2, "pube": 3, "roblock": 4}
y = tr_df["label"].astype(int).to_numpy() if "label" in tr_df.columns else tr_df["class"].map(cmap).astype(int).to_numpy()
folds = np.load(os.path.join(DATA_DIR, "folds_sgkf.npy")).astype(int)
assert len(tr_df) == len(folds) == len(y)

print("Preloading compressed JPEG bytes into RAM...")
t_load = time.time()
tr_bytes = load_jpeg_bytes(os.path.join(DATA_DIR, "train"), tr_df["file_name"].tolist())
te_bytes = load_jpeg_bytes(os.path.join(DATA_DIR, "test"), te_df["file_name"].tolist())
print(f"Preloaded train={len(tr_bytes):,}, test={len(te_bytes):,} images in {time.time()-t_load:.1f}s")

vit_freeze = FREEZE_VIT_BLOCKS if FREEZE_T4_LOWER else 0
for mkey in MODELS_TO_RUN:
    print(f"\n>>> Starting backbone [{mkey}] ({MODEL_SPECS[mkey].timm_name}) | checkpoints: {OUT_DIR}/folds_{mkey}")
    train_one_model(
        spec=MODEL_SPECS[mkey],
        tr_bytes=tr_bytes,
        te_bytes=te_bytes,
        y=y,
        folds=folds,
        tr_df=tr_df,
        te_df=te_df,
        out_dir=OUT_DIR,
        device=DEVICE,
        pretrained=True,
        epochs_override=EPOCHS_OVERRIDE,
        max_folds=MAX_FOLDS,
        num_workers=NUM_WORKERS,
        resume=True,
        freeze_vit_blocks=vit_freeze,
        p_nightmare=P_NIGHTMARE,
        p_norm_view=P_NORM_VIEW,
        seed=SEED,
    )

Preloading compressed JPEG bytes into RAM...
Preloaded train=31,546, test=25,889 images in 1.3s

>>> Starting backbone [dinov3l] (vit_large_patch16_dinov3.lvd1689m) | checkpoints: /content/drive/MyDrive/t4_hard_outputs/folds_dinov3l
[dinov3l] Config: img_hw=(192, 384) | bs=16 (accum=3, eval_bs=128) | epochs=5 | amp=torch.float16 | freeze_vit_blocks=6 | workers=2
[dinov3l] Fold 0 resumed -> ValF1=0.99090 | ExtHardF1=0.96106
[dinov3l] Fold 1 resumed -> ValF1=0.99302 | ExtHardF1=0.96869
[dinov3l] Fold 2 resumed -> ValF1=0.98999 | ExtHardF1=0.95730
[dinov3l] Fold 3 resumed -> ValF1=0.99218 | ExtHardF1=0.96362


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.21GB            

model.safetensors: downloading bytes:           |  0.00B            

[dinov3l] Fold 4 started | engine=sm75-nchw-cudnn-safe | amp=torch.float16 | trainable=227.5M/303.1M
[dinov3l] Fold 4 Ep 1/5 loss=0.5766 ValCleanF1=0.96469 | saved ckpt (Drive:fold4_ckpt.pt) (1706.7s)
[dinov3l] Fold 4 Ep 2/5 loss=0.2258 ValCleanF1=0.97111 | saved ckpt (Drive:fold4_ckpt.pt) (3361.3s)
[dinov3l] Fold 4 Ep 3/5 loss=0.1891 ValCleanF1=0.98675 | saved ckpt (Drive:fold4_ckpt.pt) (5009.4s)
[dinov3l] Fold 4 Ep 4/5 loss=0.1727 ValCleanF1=0.99017 | saved ckpt (Drive:fold4_ckpt.pt) (6652.0s)
[dinov3l] Fold 4 Ep 5/5 loss=0.1634 ValCleanF1=0.99012 | saved ckpt (Drive:fold4_ckpt.pt) (8297.1s)
[dinov3l] DONE Fold 4 in 11955.2s (eval=3658.1s) | Val6TTA F1=0.99221 | ExtHard F1=0.96390

=== [dinov3l] 5-FOLD COMPLETE | OOF_W6 F1=0.99166 LL=0.04802 | Orig2 F1=0.99204 | ExtHard F1=0.96293 | Combo=0.97729 ===

[READY] Saved 5-fold dinov3l zip -> /content/drive/MyDrive/t4_hard_outputs/t4_hard_dinov3l.zip

>>> Starting backbone [eva02l] (eva02_large_patch14_448.mim_m38m_ft_in22k_in1k) | checkpo

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

[eva02l] Fold 0 started | engine=sm75-nchw-cudnn-safe | amp=torch.float16 | trainable=227.8M/303.4M
[eva02l] Fold 0 Ep 1/4 loss=0.4634 ValCleanF1=0.97043 | saved ckpt (Drive:fold0_ckpt.pt) (2756.3s)
[eva02l] Fold 0 Ep 2/4 loss=0.2337 ValCleanF1=0.98462 | saved ckpt (Drive:fold0_ckpt.pt) (5487.7s)
[eva02l] Fold 0 Ep 3/4 loss=0.1891 ValCleanF1=0.98695 | saved ckpt (Drive:fold0_ckpt.pt) (8213.4s)
[eva02l] Fold 0 Ep 4/4 loss=0.1696 ValCleanF1=0.99079 | saved ckpt (Drive:fold0_ckpt.pt) (10932.9s)
[eva02l] DONE Fold 0 in 17382.3s (eval=6449.3s) | Val6TTA F1=0.99182 | ExtHard F1=0.96159
[eva02l] Fold 1 started | engine=sm75-nchw-cudnn-safe | amp=torch.float16 | trainable=227.8M/303.4M
[eva02l] Fold 1 Ep 1/4 loss=0.4707 ValCleanF1=0.96959 | saved ckpt (Drive:fold1_ckpt.pt) (2753.6s)
[eva02l] Fold 1 Ep 2/4 loss=0.2394 ValCleanF1=0.98672 | saved ckpt (Drive:fold1_ckpt.pt) (5487.2s)
[eva02l] Fold 1 Ep 3/4 loss=0.1858 ValCleanF1=0.98922 | saved ckpt (Drive:fold1_ckpt.pt) (8215.5s)
[eva02l] Fold 1 

: 

In [ ]:
# Cell 12: Verify All Completed Bundles & Package Combined t4_hard_bundles.zip
import glob, os, shutil, zipfile
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score, log_loss

bundle_files = sorted(glob.glob(os.path.join(OUT_DIR, "task4-*.csv")) + glob.glob(os.path.join(OUT_DIR, "task4-*.npz")))
assert bundle_files, f"No completed task4-*.csv / task4-*.npz bundles found in {OUT_DIR} yet!"

zip_path = os.path.join(OUT_DIR, "t4_hard_bundles.zip")
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for fn in bundle_files:
        zf.write(fn, os.path.basename(fn))
        if fn.endswith("hard6-oof.csv"):
            df_o = pd.read_csv(fn)
            if "label" in df_o.columns:
                y_eval = df_o["label"].astype(int).to_numpy()
            elif "y" in globals():
                y_eval = y
            else:
                tr_meta = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
                cmap_fb = {"7-11k": 0, "7-11K": 0, "fiveN": 1, "freefried": 2, "pube": 3, "roblock": 4}
                y_eval = tr_meta["class"].map(cmap_fb).astype(int).to_numpy()
            p_o = df_o[[f"prob_{i}" for i in range(5)]].to_numpy()
            p_o = p_o / np.clip(p_o.sum(axis=1, keepdims=True), 1e-12, None)
            f_clean = f1_score(y_eval, p_o.argmax(1), average="macro")
            ll_val = log_loss(y_eval, np.clip(p_o, 1e-7, 1.0), labels=[0, 1, 2, 3, 4])
            npz_match = fn.replace("-hard6-oof.csv", "-sgkf-hard-bundle.npz")
            ext_str = ""
            if os.path.isfile(npz_match):
                with np.load(npz_match) as z:
                    if "ext_w6" in z:
                        f_ext = f1_score(y_eval, z["ext_w6"].argmax(1), average="macro")
                        ext_str = f" | ExtHard F1={f_ext:.6f} | Combo={0.5*(f_clean+f_ext):.6f}"
            print(f"{os.path.basename(fn):40s} | OOF_W6 F1={f_clean:.6f} | LL={ll_val:.6f}{ext_str}")

print("Packaged combined download bundle ->", zip_path)
if os.path.isdir("/content") and os.path.abspath(zip_path) != "/content/t4_hard_bundles.zip":
    shutil.copy2(zip_path, "/content/t4_hard_bundles.zip")
    print("Copied -> /content/t4_hard_bundles.zip (1-click download in Colab sidebar)")